In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import skew, kurtosis

In [38]:
def compute_statistics(window):
    stats = {}
    for col in window.columns:
        if window[col].dtype in ['int64', 'float64']:  # Numeric columns
            stats[f'{col}_MinV'] = np.min(window[col], axis = 0)
            stats[f'{col}_MaxV'] = np.max(window[col], axis = 0)
            stats[f'{col}_std'] = np.std(window[col], axis = 0)
            stats[f'{col}_AM'] = np.mean(window[col], axis = 0)
            stats[f'{col}_AAM'] = np.mean(np.abs(window[col]), axis = 0)
            #stats[f'{col}_GM'] = np.exp(np.mean(np.log(window[col] + 1e-9), axis = 0))
            stats[f'{col}_HM'] = len(window[col]) / np.sum(1.0 / (window[col] + 1e-9), axis=0)
            stats[f'{col}_Sum'] = np.sum(window[col], axis=0)
            stats[f'{col}_Q1'] = np.percentile(window[col], 25, axis=0)
            stats[f'{col}_Median'] = np.median(window[col], axis=0)
            stats[f'{col}_Q3'] = np.percentile(window[col], 75, axis=0)
            stats[f'{col}_Variance'] = np.var(window[col], axis=0)
            #stats[f'{col}_Skewness'] = skew(window[col], axis=0)
            #stats[f'{col}_Kurtosis'] = kurtosis(window[col], axis=0)
            stats[f'{col}_Zero_Crossings'] = np.sum(np.diff(np.sign(window[col]), axis=0) != 0, axis=0)
            stats[f'{col}_Mean_Energy'] = np.mean(window[col]**2, axis=0)
            stats[f'{col}_Mean_Curve_Length'] = np.mean(np.abs(np.diff(window[col], axis=0)), axis=0)
            #stats[f'{col}_Mean_Teager_Energy'] = np.mean(window[col][1:-1]**2 - window[col][:-2] * window[col][2:], axis=0)

    return pd.Series(stats)


def sliding_window(df, window_size, step_size):
    aggregated_windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        print(window.columns)
        #aggregated_window = compute_statistics(window)
        #aggregated_windows.append(aggregated_window)
    return pd.DataFrame(aggregated_windows)

In [111]:
def remove_outliers_iqr(df):
    # Create a copy of the DataFrame to avoid modifying the original
    df_cleaned = df.copy()

    # Iterate over each column
    for column in df_cleaned.columns:
        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(df_cleaned[column]):
            # Calculate Q1 (25th percentile) and Q3 (75th percentile)
            Q1 = df_cleaned[column].quantile(0.25)
            Q3 = df_cleaned[column].quantile(0.75)
            
            # Calculate IQR (Interquartile Range)
            IQR = Q3 - Q1
            
            # Determine bounds for outliers
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Filter out rows with outliers
            df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned

In [27]:
labels_df = pd.read_csv(r"datasets\DAIC-Woz\train_split_Depression_AVEC2017.csv")
#participants = list(labels_df["Participant_ID"])
participants = [319]
print(participants)

[319]


In [17]:
display(labels_df)

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving
0,303,0,0,0,0,0,0.0,0,0,0,0,0
1,304,0,6,0,0,1,1.0,2,2,0,0,0
2,305,0,7,1,0,1,1.0,2,2,1,0,0
3,310,0,4,1,1,1,0.0,0,0,1,1,0
4,312,0,2,1,0,0,1.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
102,485,0,2,1,0,1,0.0,0,0,0,0,1
103,486,0,4,0,1,1,0.0,1,0,1,0,0
104,487,0,0,0,0,0,0.0,0,0,0,0,0
105,488,0,0,0,0,0,0.0,0,0,0,0,0


In [21]:
labels_df[labels_df["Participant_ID"] == 319].iloc[0, 1]

1

In [26]:
labels_df[labels_df["Participant_ID"] == 319]

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving
10,319,1,13,1,2,1,NaN,1,1,2,3,1


In [23]:

video_shape_list = []
audio_shape_list = []


for participant in participants:
    print(participant)
    # video
    au_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_AUs.txt")
    feat_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features.txt")
    feat3d_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features3D.txt")
    gaze_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_gaze.txt")
    pose_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_pose.txt")
    


    au_df.rename(columns = {" success" : " success0"}, inplace = True)
    feat_df.rename(columns = {" success" : " success1"}, inplace = True)
    feat3d_df.rename(columns = {" success" : " success2"}, inplace = True)
    gaze_df.rename(columns = {" success" : " success3"}, inplace = True)
    pose_df.rename(columns = {" success" : " success4"}, inplace = True)

    au_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    feat_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    feat3d_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    gaze_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    pose_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)

    video_df = pd.concat([au_df, feat_df, feat3d_df, gaze_df, pose_df], axis = 1)
    
    video_df.drop(video_df.loc[video_df[" success0"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success1"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success2"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success3"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success4"]==0].index, inplace=True)
    
    video_df.drop(columns = [" success0", " success1", " success2", " success3", " success4"], inplace = True)
    

    #video_df = remove_outliers_iqr(video_df)
    video_df.reset_index(drop = True, inplace = True)
    print("video:", video_df.shape)


    # audio
    formant_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_FORMANT.csv", header = None)
    formant_df.columns = ["F0", "F1", "F2", "F3", "F4"]
    covarep_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_COVAREP.csv", header = None)

    audio_df = pd.concat([formant_df, covarep_df], axis = 1)
    audio_df.dropna(inplace = True)

    audio_df.drop(audio_df.loc[audio_df[1]==0].index, inplace=True)
    audio_df.drop(columns=[1], inplace = True, axis = 1)

    #audio_df = remove_outliers_iqr(audio_df)
    audio_df.reset_index(drop = True, inplace = True)
    print("audio:", audio_df.shape)




    # matching the temporal rows
    #merged_df = pd.concat([video_df, audio_df], axis = 1)
    #merged_df.dropna(inplace = True)

    #video_df = merged_df.iloc[:, :378]
    
    
    
    
    
    
    # adding target label
    #video_df["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]

    #audio_df = merged_df.iloc[:, 378:]
    #audio_df["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]


    # saving csvs

    #video_df.to_csv(fr"video_matchingrows_withoutliers\{participant}_CLNF.csv", index = False)
    #audio_df.to_csv(fr"audio_matchingrows_withoutliers\{participant}.csv", index = False)



    print("video after split:", video_df.shape)
    print("audio after split:", audio_df.shape)



303
video: (29071, 378)
audio: (49986, 78)
video after split: (29071, 379)
audio after split: (29071, 79)
304
video: (21270, 378)
audio: (32463, 78)
video after split: (21270, 379)
audio after split: (21270, 79)
305
video: (49843, 378)
audio: (65130, 78)


KeyboardInterrupt: 

In [3]:
import os
import pandas as pd

In [6]:
audio_path = r"audio_matchingrows"
video_path = r"video_matchingrows"

In [ ]:
# saving audio combined csv

In [5]:
labels_df = pd.read_csv(r"datasets\DAIC-Woz\train_split_Depression_AVEC2017.csv")
participants = list(labels_df["Participant_ID"])
print(participants)

[303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341, 343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358, 360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376, 379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409, 412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433, 434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456, 457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486, 487, 488, 491]


In [7]:
complete_audio_clean_matchingrows = pd.DataFrame()

for participant in participants:
    print(participant)
    path = os.path.join(audio_path, fr"{participant}.csv")
    temp = pd.read_csv(path)


    complete_audio_clean_matchingrows = pd.concat([complete_audio_clean_matchingrows, temp], ignore_index=True, axis = 0)
    print(complete_audio_clean_matchingrows.shape)

303
(6368, 79)
304
(7981, 79)
305
(12096, 79)
310
(13535, 79)
312
(14780, 79)
313
(16333, 79)
315
(19989, 79)
316
(21806, 79)
317
(22845, 79)
318
(24197, 79)
319
(25422, 79)
320
(26846, 79)
321
(29044, 79)
322
(31920, 79)
324
(34504, 79)
325
(36622, 79)
326
(37845, 79)
327
(39633, 79)
328
(42904, 79)
330
(44100, 79)
333
(46480, 79)
336
(47933, 79)
338
(49275, 79)
339
(52573, 79)
340
(54075, 79)
341
(56617, 79)
343
(58246, 79)
344
(62198, 79)
345
(66466, 79)
347
(67918, 79)
348
(70070, 79)
350
(72424, 79)
351
(77056, 79)
352
(81131, 79)
353
(82398, 79)
355
(86590, 79)
356
(90418, 79)
357
(91120, 79)
358
(92337, 79)
360
(93969, 79)
362
(95333, 79)
363
(101067, 79)
364
(109418, 79)
366
(113468, 79)
368
(120944, 79)
369
(128455, 79)
370
(135135, 79)
371
(138221, 79)
372
(142466, 79)
374
(147342, 79)
375
(148960, 79)
376
(149989, 79)
379
(153129, 79)
380
(162130, 79)
383
(164585, 79)
385
(165433, 79)
386
(169880, 79)
391
(173180, 79)
392
(174466, 79)
393
(175568, 79)
397
(178726, 79)
400
(1

In [8]:
display(complete_audio_clean_matchingrows)

,F0,F1,F2,F3,F4,0,2,3,4,5,...,65,66,67,68,69,70,71,72,73,Target
0,823.81,1342.9,3023.7,3916.9,4888.1,185.0,0.18001,0.57174,8.50100,0.30417,...,-0.75188,-0.83839,-0.71535,-0.60432,-0.46031,-0.29668,-0.170960,0.027021,0.17660,0
1,855.41,1344.1,3023.3,3888.8,4893.2,185.0,0.17928,0.58878,8.50920,0.29284,...,-0.75995,-0.86862,-0.74154,-0.60718,-0.47623,-0.30531,-0.176540,0.027793,0.18174,0
2,824.88,1415.5,3027.1,3862.2,4884.1,185.0,0.18413,0.53192,8.20080,0.21470,...,-0.77566,-0.89961,-0.75867,-0.59685,-0.47975,-0.30743,-0.189370,0.013804,0.18516,0
3,600.05,1141.4,2149.6,3350.3,3912.6,185.5,0.15069,0.45195,8.86010,0.18683,...,-0.82941,-0.95153,-0.76394,-0.56949,-0.50288,-0.30182,-0.154040,0.062893,0.21151,0
4,329.09,1002.7,2192.3,3424.4,4402.9,186.5,0.16770,0.47168,9.79770,0.19754,...,-0.85051,-0.97877,-0.77794,-0.57748,-0.51145,-0.28415,-0.129280,0.093830,0.20739,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311098,609.00,1089.8,2765.5,4136.7,4691.4,178.0,0.11593,0.54128,4.15840,0.32831,...,-0.75282,-0.81802,-0.69638,-0.46160,-0.35066,-0.32623,-0.057107,0.151610,0.18480,0
311099,541.28,977.0,2645.0,4051.0,4322.0,176.5,0.11106,0.57947,0.65737,0.28801,...,-0.78930,-0.81728,-0.70706,-0.49656,-0.36611,-0.31995,-0.084103,0.133270,0.17458,0
311100,504.89,1044.7,2778.6,4123.1,4690.0,176.5,0.11907,0.62939,2.77580,0.30006,...,-0.82426,-0.81302,-0.72011,-0.52942,-0.38179,-0.31599,-0.104800,0.114130,0.16517,0
311101,500.00,1315.0,2740.5,3889.7,4369.6,177.5,0.13659,0.51229,8.52860,0.23565,...,-0.85542,-0.81120,-0.73206,-0.55789,-0.39403,-0.30440,-0.110940,0.102720,0.16521,0


In [9]:
complete_audio_clean_matchingrows["Target"].value_counts()

Target
0    226586
1     84517
Name: count, dtype: int64

In [10]:
complete_audio_clean_matchingrows.to_csv("train_audio_new.csv", index = False)

In [7]:
# saving video combined csv

In [11]:
complete_video_clean_matchingrows = pd.DataFrame()


for participant in participants:
    print(participant)
    path = os.path.join(video_path, fr"{participant}_CLNF.csv")
    temp = pd.read_csv(path)


    complete_video_clean_matchingrows = pd.concat([complete_video_clean_matchingrows, temp], ignore_index=True, axis = 0)
    print(complete_video_clean_matchingrows.shape)

303
(6368, 379)
304
(7981, 379)
305
(12096, 379)
310
(13535, 379)
312
(14780, 379)
313
(16333, 379)
315
(19989, 379)
316
(21806, 379)
317
(22845, 379)
318
(24197, 379)
319
(25422, 379)
320
(26846, 379)
321
(29044, 379)
322
(31920, 379)
324
(34504, 379)
325
(36622, 379)
326
(37845, 379)
327
(39633, 379)
328
(42904, 379)
330
(44100, 379)
333
(46480, 379)
336
(47933, 379)
338
(49275, 379)
339
(52573, 379)
340
(54075, 379)
341
(56617, 379)
343
(58246, 379)
344
(62198, 379)
345
(66466, 379)
347
(67918, 379)
348
(70070, 379)
350
(72424, 379)
351
(77056, 379)
352
(81131, 379)
353
(82398, 379)
355
(86590, 379)
356
(90418, 379)
357
(91120, 379)
358
(92337, 379)
360
(93969, 379)
362
(95333, 379)
363
(101067, 379)
364
(109418, 379)
366
(113468, 379)
368
(120944, 379)
369
(128455, 379)
370
(135135, 379)
371
(138221, 379)
372
(142466, 379)
374
(147342, 379)
375
(148960, 379)
376
(149989, 379)
379
(153129, 379)
380
(162130, 379)
383
(164585, 379)
385
(165433, 379)
386
(169880, 379)
391
(173180, 379)

In [12]:
display(complete_video_clean_matchingrows)

,AU01_r,AU02_r,AU04_r,AU05_r,AU06_r,AU09_r,AU10_r,AU12_r,AU14_r,AU15_r,...,x_h1,y_h1,z_h1,Tx,Ty,Tz,Rx,Ry,Rz,Target
0,0.659856,0.0,0.000000,0.000000,0.0,0.604604,0.0,0.0,0.0,0.0,...,-0.089732,0.061428,-0.994070,39.6263,57.3311,753.640,-0.041053,0.052339,-0.062683,0
1,0.845040,0.0,0.592602,0.545399,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.080333,0.049126,-0.995557,39.6560,57.3148,752.933,-0.044771,0.055062,-0.065109,0
2,0.797575,0.0,0.000000,0.672369,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.081343,0.047745,-0.995542,39.6450,57.3013,752.489,-0.047086,0.055927,-0.066365,0
3,0.926512,0.0,0.000000,0.723597,0.0,0.643866,0.0,0.0,0.0,0.0,...,-0.081569,0.048484,-0.995488,39.6569,57.2967,752.521,-0.047474,0.056318,-0.066882,0
4,0.926970,0.0,0.000000,0.695910,0.0,0.524106,0.0,0.0,0.0,0.0,...,-0.081419,0.048296,-0.995509,39.6512,57.2988,752.471,-0.047693,0.056502,-0.067129,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311098,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.297050,-0.007900,-0.954829,109.4730,91.3100,475.020,0.079525,-0.090732,0.333802,0
311099,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.305706,-0.009472,-0.952079,109.0890,91.6197,475.025,0.078601,-0.091334,0.332607,0
311100,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.295074,-0.008215,-0.955439,108.6200,91.7337,474.379,0.086557,-0.095990,0.331868,0
311101,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,...,-0.288595,0.006109,-0.957432,107.8030,91.6766,472.611,0.089232,-0.092075,0.330289,0


In [14]:
complete_video_clean_matchingrows["Target"].value_counts()

Target
0    226586
1     84517
Name: count, dtype: int64

In [15]:
complete_video_clean_matchingrows.to_csv("train_video_new.csv", index = False)